# Component 3 — Post-Integration Exploratory Data Analysis

This is the main pre-modelling EDA. It examines the integrated Western Province fisheries, chlorophyll-a, and SST data. All findings are descriptive associations. They do not establish causality or define final model features.

## Analytical safeguards

- Use only rows with both environmental variables available.
- Preserve the integrated artifacts unchanged.
- Use medians and logarithmic views because catch and chlorophyll-a are skewed.
- Show sample sizes beside grouped comparisons.
- Consolidate only obvious spelling variants for descriptive charts and save the alias evidence.
- Do not interpret repeated species rows from the same event as statistically independent evidence.
- Do not select lags, impute values, split data, or train models in this notebook.

In [ ]:
import importlib.util, subprocess, sys
packages={'pandas':'pandas>=2.0,<3','numpy':'numpy>=1.24,<3','pyarrow':'pyarrow>=14,<22','matplotlib':'matplotlib>=3.7,<4'}
missing=[spec for module,spec in packages.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
print('EDA dependencies are available.')

In [ ]:
from __future__ import annotations
import json, os, re, sys, unicodedata
from pathlib import Path
import matplotlib
if 'IPython' not in sys.modules: matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display=print
pd.set_option('display.max_columns',80); plt.style.use('seaborn-v0_8-whitegrid')
COLORS={'catch':'#4e79a7','chl':'#59a14f','sst':'#f28e2b','accent':'#8f63b8'}

## 1. Locate and load the integrated complete-case artifact

In [ ]:
def find_component_root(start:Path)->Path:
    for anchor in [start.resolve(),*start.resolve().parents]:
        for candidate in [anchor,anchor/'component - 3',anchor/'J26-DS-350'/'component - 3']:
            if (candidate/'Notebooks').is_dir() and (candidate/'src').is_dir(): return candidate.resolve()
    raise FileNotFoundError('Component 3 repository root was not found.')
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive'); ROOT=Path('/content/drive/MyDrive/FishGuard-C3')
else: ROOT=find_component_root(Path.cwd())
PROCESSED_DIR=Path(os.getenv('FISHGUARD_C3_PROCESSED_DIR',str(ROOT/'data'/'processed'))).resolve()
EVIDENCE_DIR=Path(os.getenv('FISHGUARD_C3_POST_EDA_OUTPUT',str(ROOT/'outputs'/'audit'/'post_integration_eda'))).resolve()
FIGURE_DIR=Path(os.getenv('FISHGUARD_C3_POST_EDA_FIGURES',str(ROOT/'outputs'/'figures'/'post_integration_eda'))).resolve()
for directory in [EVIDENCE_DIR,FIGURE_DIR]: directory.mkdir(parents=True,exist_ok=True)
DATA_PATH=PROCESSED_DIR/'fisheries_environment_integrated_complete.parquet'
if not DATA_PATH.is_file(): raise FileNotFoundError(f'Missing {DATA_PATH}. Run 03_data_integration.ipynb first.')
df=pd.read_parquet(DATA_PATH)
required={'event_date','latitude','longitude','species','gear','catch_weight_kg','chlorophyll_a_mg_m3','sea_surface_temperature_c','environment_complete','source_row_number'}
missing=required-set(df.columns)
if missing: raise KeyError(f'Missing integrated fields: {sorted(missing)}')
assert df.environment_complete.fillna(False).all()
df['event_date']=pd.to_datetime(df.event_date,errors='coerce'); df['year']=df.event_date.dt.year.astype('Int64'); df['month']=df.event_date.dt.month.astype('Int64')
df['log1p_catch_weight']=np.log1p(df.catch_weight_kg.clip(lower=0)); df['log1p_chlorophyll_a']=np.log1p(df.chlorophyll_a_mg_m3.clip(lower=0))
print('Rows:',f'{len(df):,}'); print('Coverage:',df.event_date.min().date(),'to',df.event_date.max().date())

## 2. Analysis-only category consolidation

In [ ]:
def category_key(value):
    value='' if pd.isna(value) else unicodedata.normalize('NFKD',str(value)).encode('ascii','ignore').decode().lower()
    return re.sub(r'[^a-z0-9]+','',value)
SPECIES_ALIASES={'skipjacktuna':'skipjack tuna','skipjack':'skipjack tuna','yellowfintuna':'yellowfin tuna','yellowfin':'yellowfin tuna','sailfish':'sailfish','swordfish':'swordfish','bigeyetuna':'bigeye tuna','frigatetuna':'frigate tuna','blackmarlin':'black marlin','bluemarlin':'blue marlin','spanishmackerel':'spanish mackerel','mobulaspp':'mobula spp'}
GEAR_ALIASES={'gillnet':'gillnet','gillnets':'gillnet','ringnet':'ring net','longline':'longline'}
def canonical_label(value,aliases):
    if pd.isna(value) or not str(value).strip(): return 'unknown'
    key=category_key(value)
    return aliases.get(key,re.sub(r'\s+',' ',str(value).strip().lower()))
df['species_eda']=df.species.map(lambda value:canonical_label(value,SPECIES_ALIASES))
df['gear_eda']=df.gear.map(lambda value:canonical_label(value,GEAR_ALIASES))
alias_summary=(df.groupby(['species','species_eda'],dropna=False).size().reset_index(name='rows').sort_values('rows',ascending=False))
gear_alias_summary=(df.groupby(['gear','gear_eda'],dropna=False).size().reset_index(name='rows').sort_values('rows',ascending=False))
alias_summary.to_csv(EVIDENCE_DIR/'species_alias_summary.csv',index=False); gear_alias_summary.to_csv(EVIDENCE_DIR/'gear_alias_summary.csv',index=False)
display(alias_summary.head(20)); display(gear_alias_summary)
print('Alias consolidation is used for EDA summaries only; source labels remain unchanged.')

## 3. Population and temporal summaries

In [ ]:
overview=pd.DataFrame([
 {'metric':'complete_integrated_rows','value':len(df)},
 {'metric':'unique_species_after_aliases','value':df.species_eda.nunique()},
 {'metric':'unique_gears_after_aliases','value':df.gear_eda.nunique()},
 {'metric':'median_catch_weight_kg','value':df.catch_weight_kg.median()},
 {'metric':'median_chlorophyll_a_mg_m3','value':df.chlorophyll_a_mg_m3.median()},
 {'metric':'median_sst_c','value':df.sea_surface_temperature_c.median()}])
annual=(df.groupby('year',observed=True).agg(records=('source_row_number','size'),median_catch_weight_kg=('catch_weight_kg','median'),median_chlorophyll_a_mg_m3=('chlorophyll_a_mg_m3','median'),median_sst_c=('sea_surface_temperature_c','median')).reset_index())
monthly=(df.groupby('month',observed=True).agg(records=('source_row_number','size'),median_catch_weight_kg=('catch_weight_kg','median'),median_chlorophyll_a_mg_m3=('chlorophyll_a_mg_m3','median'),median_sst_c=('sea_surface_temperature_c','median')).reset_index())
overview.to_csv(EVIDENCE_DIR/'post_integration_overview.csv',index=False); annual.to_csv(EVIDENCE_DIR/'annual_environment_catch_summary.csv',index=False); monthly.to_csv(EVIDENCE_DIR/'monthly_environment_catch_summary.csv',index=False)
display(overview); display(annual)

In [ ]:
fig,axes=plt.subplots(3,1,figsize=(12,11),sharex=True)
axes[0].plot(monthly.month,monthly.median_catch_weight_kg,marker='o',color=COLORS['catch']); axes[0].set_ylabel('Median catch (kg)'); axes[0].set_title('Monthly descriptive patterns')
axes[1].plot(monthly.month,monthly.median_chlorophyll_a_mg_m3,marker='o',color=COLORS['chl']); axes[1].set_ylabel('Median chlorophyll-a')
axes[2].plot(monthly.month,monthly.median_sst_c,marker='o',color=COLORS['sst']); axes[2].set_ylabel('Median SST (°C)'); axes[2].set_xlabel('Calendar month'); axes[2].set_xticks(range(1,13))
for ax in axes: ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'monthly_catch_environment_patterns.png',dpi=200,bbox_inches='tight'); plt.show()

## 4. Catch–environment associations

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(15,6))
h1=axes[0].hexbin(df.log1p_chlorophyll_a,df.log1p_catch_weight,gridsize=45,mincnt=1,cmap='Greens',bins='log'); axes[0].set_xlabel('log1p(chlorophyll-a)'); axes[0].set_ylabel('log1p(catch weight kg)'); axes[0].set_title('Catch weight and chlorophyll-a'); fig.colorbar(h1,ax=axes[0],label='log10(record count)')
h2=axes[1].hexbin(df.sea_surface_temperature_c,df.log1p_catch_weight,gridsize=45,mincnt=1,cmap='Oranges',bins='log'); axes[1].set_xlabel('SST (°C)'); axes[1].set_ylabel('log1p(catch weight kg)'); axes[1].set_title('Catch weight and SST'); fig.colorbar(h2,ax=axes[1],label='log10(record count)')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'catch_environment_density.png',dpi=200,bbox_inches='tight'); plt.show()

In [ ]:
df['chlorophyll_quintile']=pd.qcut(df.chlorophyll_a_mg_m3,5,labels=False,duplicates='drop')+1
df['sst_quintile']=pd.qcut(df.sea_surface_temperature_c,5,labels=False,duplicates='drop')+1
chl_bins=(df.groupby('chlorophyll_quintile').agg(records=('source_row_number','size'),environment_min=('chlorophyll_a_mg_m3','min'),environment_max=('chlorophyll_a_mg_m3','max'),median_catch_weight_kg=('catch_weight_kg','median')).reset_index())
sst_bins=(df.groupby('sst_quintile').agg(records=('source_row_number','size'),environment_min=('sea_surface_temperature_c','min'),environment_max=('sea_surface_temperature_c','max'),median_catch_weight_kg=('catch_weight_kg','median')).reset_index())
chl_bins.insert(0,'variable','chlorophyll_a'); sst_bins.insert(0,'variable','sst')
environment_bins=pd.concat([chl_bins.rename(columns={'chlorophyll_quintile':'quintile'}),sst_bins.rename(columns={'sst_quintile':'quintile'})],ignore_index=True)
environment_bins.to_csv(EVIDENCE_DIR/'environment_quintile_summary.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(14,5))
axes[0].plot(chl_bins.chlorophyll_quintile,chl_bins.median_catch_weight_kg,marker='o',color=COLORS['chl']); axes[0].set_title('Median catch across chlorophyll-a quintiles'); axes[0].set_xlabel('Chlorophyll-a quintile'); axes[0].set_ylabel('Median catch weight (kg)')
axes[1].plot(sst_bins.sst_quintile,sst_bins.median_catch_weight_kg,marker='o',color=COLORS['sst']); axes[1].set_title('Median catch across SST quintiles'); axes[1].set_xlabel('SST quintile'); axes[1].set_ylabel('Median catch weight (kg)')
for ax in axes: ax.set_xticks(range(1,6)); ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'environment_quintile_catch.png',dpi=200,bbox_inches='tight'); plt.show()

## 5. Rank correlations

In [ ]:
correlation_fields=['log1p_catch_weight','log1p_chlorophyll_a','sea_surface_temperature_c','latitude','longitude','month','year']
correlations=df[correlation_fields].corr(method='spearman')
correlations.to_csv(EVIDENCE_DIR/'spearman_correlations.csv')
fig,ax=plt.subplots(figsize=(9,8)); image=ax.imshow(correlations,vmin=-1,vmax=1,cmap='RdBu_r')
ax.set_xticks(range(len(correlation_fields)),correlation_fields,rotation=45,ha='right'); ax.set_yticks(range(len(correlation_fields)),correlation_fields)
for i in range(len(correlation_fields)):
    for j in range(len(correlation_fields)): ax.text(j,i,f'{correlations.iloc[i,j]:.2f}',ha='center',va='center',fontsize=9)
ax.set_title('Spearman rank correlations'); fig.colorbar(image,ax=ax,label='Spearman correlation')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'spearman_correlation_matrix.png',dpi=200,bbox_inches='tight'); plt.show()

## 6. Species and gear environmental profiles

In [ ]:
species_profiles=(df.groupby('species_eda').agg(records=('source_row_number','size'),median_catch_weight_kg=('catch_weight_kg','median'),median_chlorophyll_a_mg_m3=('chlorophyll_a_mg_m3','median'),median_sst_c=('sea_surface_temperature_c','median')).reset_index().sort_values('records',ascending=False))
gear_profiles=(df.groupby('gear_eda').agg(records=('source_row_number','size'),median_catch_weight_kg=('catch_weight_kg','median'),median_chlorophyll_a_mg_m3=('chlorophyll_a_mg_m3','median'),median_sst_c=('sea_surface_temperature_c','median')).reset_index().sort_values('records',ascending=False))
species_profiles.to_csv(EVIDENCE_DIR/'species_environment_profiles.csv',index=False); gear_profiles.to_csv(EVIDENCE_DIR/'gear_environment_profiles.csv',index=False)
top=species_profiles.head(12).sort_values('records')
fig,axes=plt.subplots(1,3,figsize=(19,7),sharey=True)
axes[0].barh(top.species_eda,top.median_catch_weight_kg,color=COLORS['catch']); axes[0].set_xlabel('Median catch (kg)'); axes[0].set_title('Catch')
axes[1].barh(top.species_eda,top.median_chlorophyll_a_mg_m3,color=COLORS['chl']); axes[1].set_xlabel('Median chlorophyll-a'); axes[1].set_title('Chlorophyll-a')
axes[2].barh(top.species_eda,top.median_sst_c,color=COLORS['sst']); axes[2].set_xlabel('Median SST (°C)'); axes[2].set_title('SST')
fig.suptitle('Profiles of the most frequently recorded species categories'); fig.tight_layout(); fig.savefig(FIGURE_DIR/'species_environment_profiles.png',dpi=200,bbox_inches='tight'); plt.show()
display(gear_profiles)

## 7. Spatial patterns

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(20,6),sharex=True,sharey=True)
specs=[('catch_weight_kg',np.median,'Median catch weight (kg)','Blues'),('chlorophyll_a_mg_m3',np.median,'Median chlorophyll-a','Greens'),('sea_surface_temperature_c',np.median,'Median SST (°C)','Oranges')]
for ax,(field,reducer,title,cmap) in zip(axes,specs):
    result=ax.hexbin(df.longitude,df.latitude,C=df[field],reduce_C_function=reducer,gridsize=35,mincnt=1,cmap=cmap); ax.set_title(title); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); fig.colorbar(result,ax=ax)
fig.tight_layout(); fig.savefig(FIGURE_DIR/'spatial_catch_environment_patterns.png',dpi=200,bbox_inches='tight'); plt.show()

## 8. Save viva findings and reproducibility manifest

In [ ]:
catch_corr_chl=correlations.loc['log1p_catch_weight','log1p_chlorophyll_a']; catch_corr_sst=correlations.loc['log1p_catch_weight','sea_surface_temperature_c']
findings=pd.DataFrame([
 {'finding':'Analytical population','result':f'{len(df):,} complete integrated observations'},
 {'finding':'Catch-weight distribution','result':'Strongly right-skewed; use robust summaries and consider log1p target transformation'},
 {'finding':'Catch and chlorophyll-a rank correlation','result':f'{catch_corr_chl:.3f}'},
 {'finding':'Catch and SST rank correlation','result':f'{catch_corr_sst:.3f}'},
 {'finding':'Most recorded species after alias consolidation','result':str(species_profiles.iloc[0].species_eda)},
 {'finding':'Most recorded gear after alias consolidation','result':str(gear_profiles.iloc[0].gear_eda)},
 {'finding':'Interpretation limit','result':'Associations may reflect year, month, location, species, gear, vessel, and sampling differences; no causal claim'}])
findings.to_csv(EVIDENCE_DIR/'viva_findings.csv',index=False)
manifest={'created_utc':pd.Timestamp.now(tz='UTC').isoformat(),'runtime':'Google Colab' if IN_COLAB else 'Local Windows','input':DATA_PATH.name,'rows':len(df),'analysis_scope':'descriptive post-integration EDA','source_artifact_modified':False,'category_aliases_analysis_only':True,'figures':sorted(p.name for p in FIGURE_DIR.glob('*.png')),'evidence':sorted(p.name for p in EVIDENCE_DIR.glob('*.csv'))}
with (EVIDENCE_DIR/'post_integration_eda_manifest.json').open('w',encoding='utf-8') as f: json.dump(manifest,f,indent=2)
display(findings); print('Evidence:',EVIDENCE_DIR); print('Figures:',FIGURE_DIR)

## Viva interpretation

Use this notebook to explain how catch, season, location, species, gear, chlorophyll-a, and SST vary together in the available records. Emphasize that the observations are uneven across years and categories, environmental values repeat for multiple species rows from the same fishing event, and simple correlations can be confounded. Final predictors, lag windows, validation splits, transformations, and leakage controls must be decided in later notebooks.